In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, split

try:
    spark.stop()
except Exception:
    pass

spark = SparkSession.builder \
    .appName("Analisis_BigData_Final") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/TiendaBigData.AmazonLaptops") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
print(f"Registros cargados: {df.count()}")
df.show(5, truncate=False)

In [ ]:
df_transformado = df.withColumn("marca", split(col("identificador"), " ").getItem(0))
df_validado = df_transformado.filter(col("valor") > 100)

reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

reporte_marcas.show(truncate=False)

In [ ]:
top_productos = df_validado.orderBy(col("valor").desc())
top_productos.select("identificador", "valor", "grupo", "fecha_captura").show(10, truncate=False)